# Market basket analysis on a local Julia kernel

A retailer wants to know which products customers buy together. These cells clean the receipts of Online Retail II into one basket per invoice, mine them with the team's Eclat, derive association rules, and show the rules with the highest lift. Open the notebook from a checkout of the repository in an IDE with a Julia kernel; the first two cells find the checkout and install the project. The cleaning downloads about 95 MB, and the mining takes about three minutes.

The same steps run on Google Colab in `market_basket_colab.ipynb`, which differs only in setup. `applications/market_basket/README.md` explains each cleaning step, and `docs/market-basket-rules-2026-10-29.md` records the results.

In [ ]:
# The notebook sits in notebooks/, so the checkout is its parent unless the working directory already is the checkout.
ROOT = abspath(get(ENV, "ITEMSET_ROOT", isfile("Project.toml") ? pwd() : dirname(pwd())))
isfile(joinpath(ROOT, "Project.toml")) ||
    error("Open the notebook from the repository root or from notebooks/, or set ITEMSET_ROOT to the checkout")
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", ispath(".git") ? readchomp(`git rev-parse HEAD`) : "no .git folder; record the commit of this copy")


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Use Julia 1.12 or later within 1.x; 1.13.2 gives the recorded results")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())
artifact_manifest = get(ENV, "ITEMSET_ARTIFACT_MANIFEST", "")
notebook_artifacts = Dict{String,String}()


In [ ]:
# Clean Online Retail II into invoice baskets (about 95 MB download).
module PrepareBaskets
    include(joinpath(Main.ROOT, "applications", "market_basket", "prepare.jl"))
end
PrepareBaskets.main(String[])
items = split.(readlines(joinpath(ROOT, "data", "raw", "market_basket", "items.tsv"))[2:end], '\t')
first(sort(items; by=row -> -parse(Int, row[4])), 10)


In [ ]:
# Mine the baskets, derive association rules, and check them (about three minutes).
module MarketBasketRules
    include(joinpath(Main.ROOT, "applications", "market_basket", "analyze.jl"))
end
MarketBasketRules.main(String[])
notebook_artifacts["market_basket"] = joinpath(ROOT, "results", "raw", "market_basket")
table(name) = split.(readlines(joinpath(ROOT, "results", "raw", "market_basket", name)), '\t')
table("top_groups.tsv")


In [ ]:
# Rule measures on baskets small enough to check by hand. The formulas are the ones that rank the real baskets above.
# Seven baskets: {2} {2} {1,2} {1,2,3} {1,2,3} {2,3} {1,2,3}, so N = 7.
# For the rule 1 => 2,3, by hand: support(1,2,3) = 3, support(1) = 4, support(2,3) = 4.
#   confidence = 3/4 and lift = (3/4) / (4/7) = 21/16.
# At minconf 0.7 the rule passes, and at 0.76 it fails because 3/4 < 0.76, while 1,3 => 2 (confidence 1) still passes.
module HandCheckedRules
    using ItemsetMining
    include(joinpath(Main.ROOT, "applications", "market_basket", "MarketBasket.jl"))
    using .MarketBasket: association_rules, confidence, lift

    baskets = [[2], [2], [1, 2], [1, 2, 3], [1, 2, 3], [2, 3], [1, 2, 3]]
    patterns = mine(baskets; minsup=1)
    find(rules, antecedent, consequent) =
        filter(r -> r.antecedent == antecedent && r.consequent == consequent, rules)

    rules = association_rules(patterns, length(baskets); minconf=0.7)
    rule = only(find(rules, (1,), (2, 3)))
    @assert (rule.support, rule.antecedent_support, rule.consequent_support) == (3, 4, 4)
    @assert confidence(rule) == 3 // 4 && lift(rule) == 21 // 16
    println("1 => 2,3: support ", rule.support, ", confidence ", confidence(rule), ", lift ", lift(rule))

    strict = association_rules(patterns, length(baskets); minconf=0.76)
    @assert isempty(find(strict, (1,), (2, 3)))
    @assert !isempty(find(strict, (1, 3), (2,))) && confidence(only(find(strict, (1, 3), (2,)))) == 1
    println("minconf 0.7 keeps ", length(rules), " rules; minconf 0.76 keeps ", length(strict), " and drops 1 => 2,3")
end